# Fine-tune a detector on COCO

Baseline run before the augmentation change.


In [1]:
%matplotlib inline
!pip install -q torch torchvision

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from data.dataset import CocoDetection


## The model


In [2]:
class Detector(nn.Module):
    def __init__(self, num_classes=80):
        super().__init__()
        self.backbone = nn.Sequential(nn.Conv2d(3, 64, 3), nn.ReLU())
        self.head = nn.Linear(64, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


## Transforms


In [3]:
def build_transforms(train=True):
    ops = [T.ToTensor()]
    if train:
        ops.append(T.RandomHorizontalFlip())
    return T.Compose(ops)


## Train


In [4]:
train_set = CocoDetection("data/coco", transforms=build_transforms(True))
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)

model = Detector(num_classes=80)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(10):
    for images, targets in train_loader:
        loss = model(images).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    torch.save(model.state_dict(), "runs/last.pt")


## Evaluate mAP


In [5]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0.0
    for images, targets in loader:
        total += float(model(images).sum())
    return {"val/mAP": total / max(len(loader), 1)}


RuntimeError: CUDA out of memory

In [6]:
checkpoint = torch.load("runs/last.pt")
model.load_state_dict(checkpoint)
stats = evaluate(model, train_loader)
print(stats["val/mAP"])
